# Smoking model trajectories, equilibria, eigenvalues, sensitivity indices, and validation checks Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

- `https://sen.science/doi/10.71728/senscience.vd7h-gtrs/fair2.json`

### Goal
We will:
- load the dataset metadata,
- inspect Croissant entities using their `@id` values,
- discover available record sets and fields,
- load records into pandas DataFrames,
- perform simple exploratory data analysis,
- create a few visualizations.

All dataset entities should be referenced by their Croissant `@id` wherever possible.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant`.

In this section, we keep the dataset URL in a variable so the notebook can be reused easily with another Croissant file.

In [ ]:
import json
import warnings

import mlcroissant as mlc
import pandas as pd

warnings.filterwarnings("ignore")

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vd7h-gtrs/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object, then convert to JSON-like structure for inspection
metadata_obj = dataset.metadata
metadata_json = metadata_obj.to_json()

print("Dataset loaded successfully.\n")
print(f"Name: {metadata_json.get('name', 'N/A')}")
print(f"Description: {metadata_json.get('description', 'N/A')}")
print(f"Identifier: {metadata_json.get('identifier', 'N/A')}")
print(f"Version: {metadata_json.get('version', 'N/A')}")
print(f"License: {metadata_json.get('license', 'N/A')}")
print(f"Temporal coverage: {metadata_json.get('temporalCoverage', 'N/A')}")

In [ ]:
# Display a compact metadata summary
summary_fields = [
    "@id",
    "name",
    "description",
    "citeAs",
    "datePublished",
    "keywords",
    "distribution",
    "recordSet",
]

for field_name in summary_fields:
    value = metadata_json.get(field_name, None)
    if isinstance(value, list):
        print(f"{field_name}: list with {len(value)} item(s)")
    else:
        print(f"{field_name}: {value}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Because Croissant datasets are organized around entities like record sets and fields, it is useful to inspect the metadata structure before attempting extraction.

The code below prints the dataset-level `recordSet` entries from the Croissant JSON. If the dataset exposes record sets through the loaded metadata, we will capture their `@id` values dynamically.

In [ ]:
# Inspect record sets declared in the metadata JSON
record_sets_meta = metadata_json.get("recordSet", [])
print(f"Number of record sets declared in metadata: {len(record_sets_meta)}\n")

for idx, rs in enumerate(record_sets_meta, start=1):
    rs_id = rs.get("@id", f"record_set_{idx}") if isinstance(rs, dict) else f"record_set_{idx}"
    rs_name = rs.get("name", "N/A") if isinstance(rs, dict) else "N/A"
    print(f"{idx}. @id: {rs_id}")
    print(f"   name: {rs_name}")
    if isinstance(rs, dict) and "field" in rs:
        fields = rs.get("field", [])
        print(f"   fields: {len(fields)}")
        for field in fields[:10]:
            if isinstance(field, dict):
                print(f"      - field @id: {field.get('@id', 'N/A')} | name: {field.get('name', 'N/A')} | dataType: {field.get('dataType', 'N/A')}")
        if len(fields) > 10:
            print("      ...")
    print()

In [ ]:
# Build a DataFrame summarizing record sets and fields by @id where available
record_set_rows = []
field_rows = []

for rs in record_sets_meta:
    if not isinstance(rs, dict):
        continue
    rs_id = rs.get("@id", None)
    record_set_rows.append({
        "record_set_@id": rs_id,
        "record_set_name": rs.get("name", None),
        "record_set_description": rs.get("description", None),
        "field_count": len(rs.get("field", [])) if isinstance(rs.get("field", []), list) else 0,
    })
    for field in rs.get("field", []):
        if not isinstance(field, dict):
            continue
        field_rows.append({
            "record_set_@id": rs_id,
            "field_@id": field.get("@id", None),
            "field_name": field.get("name", None),
            "field_description": field.get("description", None),
            "field_dataType": field.get("dataType", None),
            "field_source": field.get("source", None),
        })

record_sets_df = pd.DataFrame(record_set_rows)
fields_df = pd.DataFrame(field_rows)

print("Record set summary:")
display(record_sets_df)

print("Field summary:")
display(fields_df.head(50))

If record set `@id` values are available, the next cell attempts to preview records from the first record set.

This follows the standard `mlcroissant` pattern:

- call `dataset.records(record_set=<record_set_@id>)`
- iterate over records
- inspect the resulting dictionaries.

In [ ]:
# Preview records from the first discovered record set, if available
record_set_ids = record_sets_df["record_set_@id"].dropna().tolist() if not record_sets_df.empty else []

if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"Previewing up to 3 records from record set @id: {example_record_set_id}\n")
    try:
        for i, x in enumerate(dataset.records(record_set=example_record_set_id)):
            print(x)
            print()
            if i >= 2:
                break
    except Exception as e:
        print(f"Could not iterate records for record set {example_record_set_id}: {e}")
else:
    print("No record set @id values were declared in metadata_json['recordSet'].")
    print("This may mean the Croissant file exposes distributions and dataset-level metadata, but not explicit recordSet entries in the loaded metadata.")

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis.

This section is written to work dynamically:
- it collects all discovered record set `@id`s,
- tries to load records for each one,
- stores the results in a dictionary keyed by the record set `@id`.

If the dataset does not expose record sets directly, the code will fail gracefully and report what happened.

In [ ]:
# Extract data from each discovered record set
dataframes = {}
load_report = []

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        load_report.append({
            "record_set_@id": record_set_id,
            "status": "loaded",
            "rows": len(df),
            "columns": len(df.columns),
        })
    except Exception as e:
        load_report.append({
            "record_set_@id": record_set_id,
            "status": f"failed: {e}",
            "rows": None,
            "columns": None,
        })

load_report_df = pd.DataFrame(load_report)
display(load_report_df)

In [ ]:
# Show columns and sample rows for the first successfully loaded record set
loaded_record_set_ids = [k for k, v in dataframes.items() if isinstance(v, pd.DataFrame)]

if loaded_record_set_ids:
    sample_record_set_id = loaded_record_set_ids[0]
    print(f"First loaded record set @id: {sample_record_set_id}")
    print("Columns:")
    print(dataframes[sample_record_set_id].columns.tolist())
    display(dataframes[sample_record_set_id].head())
else:
    print("No record sets were successfully loaded into DataFrames.")
    print("You can still use the metadata inspection above to understand the dataset structure.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as:
- identifying numeric fields,
- filtering records,
- normalizing a numeric field,
- grouping by a categorical field when available.

Since the exact record set and field `@id`s may vary depending on how the Croissant file is exposed, this section selects a usable DataFrame dynamically.

In [ ]:
import numpy as np

# Select one loaded record set for EDA
eda_record_set_id = loaded_record_set_ids[0] if loaded_record_set_ids else None

if eda_record_set_id is not None:
    eda_df = dataframes[eda_record_set_id].copy()
    print(f"Using record set @id for EDA: {eda_record_set_id}")
    print(f"Shape: {eda_df.shape}")
    display(eda_df.head())
else:
    eda_df = pd.DataFrame()
    print("No DataFrame is available for EDA.")

In [ ]:
# Identify numeric and non-numeric columns
if not eda_df.empty:
    numeric_columns = eda_df.select_dtypes(include=[np.number]).columns.tolist()
    non_numeric_columns = eda_df.select_dtypes(exclude=[np.number]).columns.tolist()

    print("Numeric columns:")
    print(numeric_columns)
    print("\nNon-numeric columns:")
    print(non_numeric_columns)

    if numeric_columns:
        display(eda_df[numeric_columns].describe().T)
    else:
        print("No numeric columns detected in the selected record set.")
else:
    print("EDA skipped because no DataFrame is available.")

In [ ]:
# Example filtering, normalization, and grouping using dynamically selected fields
if not eda_df.empty:
    numeric_columns = eda_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_candidates = [c for c in eda_df.columns if c not in numeric_columns]

    if numeric_columns:
        numeric_field_id = numeric_columns[0]
        threshold = eda_df[numeric_field_id].median()
        filtered_df = eda_df[eda_df[numeric_field_id] > threshold].copy()

        print(f"Selected numeric field: {numeric_field_id}")
        print(f"Threshold (median): {threshold}")
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        std_value = filtered_df[numeric_field_id].std()
        if pd.notnull(std_value) and std_value != 0:
            normalized_column_name = f"{numeric_field_id}_normalized"
            filtered_df[normalized_column_name] = (
                (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
                filtered_df[numeric_field_id].std()
            )
            print(f"Normalized values for field: {numeric_field_id}")
            display(filtered_df[[numeric_field_id, normalized_column_name]].head())
        else:
            print(f"Could not normalize {numeric_field_id} because the standard deviation is zero or undefined.")

        if categorical_candidates:
            group_field_id = categorical_candidates[0]
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_columns].mean(numeric_only=True)
            display(grouped_df.head())
        else:
            print("No categorical/grouping field was available.")
    else:
        print("No numeric fields available for filtering and normalization.")
else:
    print("No DataFrame is available for EDA operations.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset.

The plots below are generated only if the selected record set contains suitable numeric columns.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

if not eda_df.empty:
    numeric_columns = eda_df.select_dtypes(include=[np.number]).columns.tolist()

    if len(numeric_columns) >= 1:
        plt.figure(figsize=(8, 5))
        sns.histplot(eda_df[numeric_columns[0]].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_columns[0]}")
        plt.xlabel(numeric_columns[0])
        plt.ylabel("Count")
        plt.show()

    if len(numeric_columns) >= 2:
        plt.figure(figsize=(7, 5))
        sns.scatterplot(data=eda_df, x=numeric_columns[0], y=numeric_columns[1])
        plt.title(f"{numeric_columns[0]} vs {numeric_columns[1]}")
        plt.xlabel(numeric_columns[0])
        plt.ylabel(numeric_columns[1])
        plt.show()

    if len(numeric_columns) >= 2:
        corr_df = eda_df[numeric_columns].corr(numeric_only=True)
        plt.figure(figsize=(8, 6))
        sns.heatmap(corr_df, annot=True, cmap="coolwarm", fmt=".2f")
        plt.title("Correlation Heatmap")
        plt.show()
    elif len(numeric_columns) == 1:
        print("Only one numeric column is available, so correlation heatmap is skipped.")
    else:
        print("No numeric columns are available for visualization.")
else:
    print("Visualization skipped because no DataFrame is available.")

## 6. Dataset Distributions and File Objects
For Croissant datasets, it is often useful to inspect the declared file distributions as part of understanding how data is packaged.

Below, we extract the distribution `@id` values from the metadata.

In [ ]:
distribution_entries = metadata_json.get("distribution", [])

distribution_rows = []
for item in distribution_entries:
    if isinstance(item, dict):
        distribution_rows.append({
            "distribution_@id": item.get("@id", None),
            "name": item.get("name", None),
            "contentUrl": item.get("contentUrl", None),
            "encodingFormat": item.get("encodingFormat", None),
            "sha256": item.get("sha256", None),
        })

distribution_df = pd.DataFrame(distribution_rows)
display(distribution_df)

## 7. Conclusion
In this notebook, we:
- loaded a Croissant dataset with `mlcroissant`,
- accessed dataset metadata safely through a metadata object,
- inspected dataset-level `@id` metadata,
- attempted to discover and load record sets dynamically,
- performed basic EDA and visualization when records were available,
- reviewed distribution objects by their `@id` values.

This dataset describes a reproducibility package for a smoking and e-cigarette modeling workflow, including simulation trajectories, equilibrium outputs, eigenvalues, sensitivity indices, and validation checks.

If explicit Croissant `recordSet` definitions become available in the schema, the same notebook structure can be reused directly by plugging in the record set `@id` values and field `@id` values discovered during the metadata overview step.